# Notebook 04 — Strategic Retention Playbook

Identifies the highest-risk customer segments and maps each to concrete retention actions.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from skills.utils.plotting import save_fig
from skills.utils.segment_helpers import compute_churn_rate, segment_summary

sns.set_theme(style='whitegrid')


## Load Data & Pre-computed Results

In [2]:
df = pd.read_csv('../Customer-Churn-Records.csv')
cross_summary = pd.read_csv('../outputs/tables/cross_segment_summary.csv')
lr_coef = pd.read_csv('../outputs/tables/lr_coefficients.csv')
fi = pd.read_csv('../outputs/tables/feature_importance.csv')
overall_churn = df['Exited'].mean()
print(f"Overall churn rate: {overall_churn:.1%}")


Overall churn rate: 20.4%


## Top 5 Highest-Risk Segments

In [3]:
top_segments = cross_summary.sort_values('churn_rate', ascending=False).head(5)
print("Top 5 Highest-Risk Segments:")
display_cols = ['Card Type', 'Geography', 'churn_rate', 'n_customers', 'avg_satisfaction', 'complaint_rate']
print(top_segments[display_cols].to_string(index=False))


Top 5 Highest-Risk Segments:
Card Type Geography  churn_rate  n_customers  avg_satisfaction  complaint_rate
  DIAMOND   Germany       0.340          648          3.038580        0.339506
 PLATINUM   Germany       0.337          608          3.009868        0.340461
   SILVER   Germany       0.313          600          2.976667        0.313333
     GOLD   Germany       0.308          653          2.996937        0.312404
  DIAMOND    France       0.180         1230          2.957724        0.180488


In [4]:
fig, ax = plt.subplots(figsize=(11, 6))
labels = top_segments['Card Type'] + '\n(' + top_segments['Geography'] + ')'
bars = ax.bar(labels, top_segments['churn_rate'] * 100, color='#E53935', edgecolor='white')
ax.axhline(overall_churn * 100, color='steelblue', linestyle='--', linewidth=1.5,
           label=f'Overall avg ({overall_churn:.1%})')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Top 5 Highest-Risk Customer Segments', fontsize=13)
ax.legend()
for bar, val in zip(bars, top_segments['churn_rate'] * 100):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
save_fig(fig, '04_top_risk_segments.png', output_dir='../outputs/figures')
plt.close(fig)


## Top Churn Drivers (from Logistic Regression)

In [5]:
top_positive = lr_coef[lr_coef['coefficient'] > 0].sort_values('coefficient', ascending=False).head(5)
top_negative = lr_coef[lr_coef['coefficient'] < 0].sort_values('coefficient').head(5)
print("Top factors INCREASING churn risk:")
print(top_positive[['feature', 'coefficient']].to_string(index=False))
print("\nTop factors DECREASING churn risk (protective):")
print(top_negative[['feature', 'coefficient']].to_string(index=False))


Top factors INCREASING churn risk:
         feature  coefficient
        Complain     5.183796
             Age     0.854531
Card Type_SILVER     0.109805
Geography_France     0.072552
     CreditScore     0.065510

Top factors DECREASING churn risk (protective):
           feature  coefficient
    IsActiveMember    -0.621829
      Point Earned    -0.395773
Satisfaction Score    -0.208162
 Geography_Germany    -0.118994
     NumOfProducts    -0.085284


## Strategic Retention Playbook

In [6]:
recommendations = [
    {
        'Theme': 'High Complaint Rate',
        'Key Driver': 'Complain (strongest churn predictor in LR)',
        'Target Segments': 'All segments — especially SILVER/GOLD in Germany',
        'Actions': [
            'Launch proactive outreach for all customers with any complaint history: assign a dedicated relationship manager for 30 days after a complaint.',
            'Implement a complaint-resolution SLA (48h escalation) + automatic follow-up with satisfaction survey and a goodwill gesture (fee waiver or bonus points).',
        ]
    },
    {
        'Theme': 'Inactive Members',
        'Key Driver': 'IsActiveMember = 0 (strong negative predictor)',
        'Target Segments': 'Any card type with >60 days of inactivity',
        'Actions': [
            'Trigger a re-engagement email + app push notification after 60 days of inactivity, offering a 2x points multiplier for the next 30 days.',
            'For inactive high-balance customers, offer a free annual financial health review to surface better-fit products.',
        ]
    },
    {
        'Theme': 'Customers with 3+ Products',
        'Key Driver': 'NumOfProducts >= 3 shows high churn in EDA',
        'Target Segments': 'Any segment where NumOfProducts = 3 or 4',
        'Actions': [
            'Audit product bundling — customers with 3+ products may feel over-sold; offer a free annual portfolio review to right-size.',
            'Create a loyalty tier benefit for multi-product holders (priority service, higher savings rates) to increase perceived value.',
        ]
    },
    {
        'Theme': 'GOLD & SILVER Cards in Germany',
        'Key Driver': 'Highest churn rate + complaint rate in Germany cross-segment',
        'Target Segments': 'GOLD Germany, SILVER Germany',
        'Actions': [
            'Run a targeted card upgrade campaign (GOLD→PLATINUM or SILVER→GOLD) with reduced annual fee for the first year.',
            'Survey recent churners in Germany to surface market-specific pain points and address top 3 issues within one quarter.',
        ]
    },
    {
        'Theme': 'Low Satisfaction Score',
        'Key Driver': 'Satisfaction Score (protective when high, risky when low)',
        'Target Segments': 'Any segment with avg satisfaction score < 3',
        'Actions': [
            'Implement quarterly NPS surveys; flag customers scoring ≤6 for immediate retention team follow-up.',
            'Create a Satisfaction Guarantee: customers dissatisfied with a product in first 90 days get a no-penalty switch to an alternative product.',
        ]
    },
]

for rec in recommendations:
    print(f"\n{'='*65}")
    print(f"THEME:          {rec['Theme']}")
    print(f"KEY DRIVER:     {rec['Key Driver']}")
    print(f"TARGET:         {rec['Target Segments']}")
    print(f"ACTIONS:")
    for i, action in enumerate(rec['Actions'], 1):
        print(f"  {i}. {action}")



THEME:          High Complaint Rate
KEY DRIVER:     Complain (strongest churn predictor in LR)
TARGET:         All segments — especially SILVER/GOLD in Germany
ACTIONS:
  1. Launch proactive outreach for all customers with any complaint history: assign a dedicated relationship manager for 30 days after a complaint.
  2. Implement a complaint-resolution SLA (48h escalation) + automatic follow-up with satisfaction survey and a goodwill gesture (fee waiver or bonus points).

THEME:          Inactive Members
KEY DRIVER:     IsActiveMember = 0 (strong negative predictor)
TARGET:         Any card type with >60 days of inactivity
ACTIONS:
  1. Trigger a re-engagement email + app push notification after 60 days of inactivity, offering a 2x points multiplier for the next 30 days.
  2. For inactive high-balance customers, offer a free annual financial health review to surface better-fit products.

THEME:          Customers with 3+ Products
KEY DRIVER:     NumOfProducts >= 3 shows high churn in 

## Export Playbook Table

In [7]:
rows = []
for rec in recommendations:
    for action in rec['Actions']:
        rows.append({
            'Theme': rec['Theme'],
            'Key Driver': rec['Key Driver'],
            'Target Segments': rec['Target Segments'],
            'Recommended Action': action,
        })
playbook_df = pd.DataFrame(rows)
playbook_df.to_csv('../outputs/tables/retention_playbook.csv', index=False)
print(f"Retention playbook saved: {len(playbook_df)} action rows")
print(playbook_df[['Theme', 'Target Segments']].to_string(index=False))


Retention playbook saved: 10 action rows
                         Theme                                  Target Segments
           High Complaint Rate All segments — especially SILVER/GOLD in Germany
           High Complaint Rate All segments — especially SILVER/GOLD in Germany
              Inactive Members        Any card type with >60 days of inactivity
              Inactive Members        Any card type with >60 days of inactivity
    Customers with 3+ Products         Any segment where NumOfProducts = 3 or 4
    Customers with 3+ Products         Any segment where NumOfProducts = 3 or 4
GOLD & SILVER Cards in Germany                     GOLD Germany, SILVER Germany
GOLD & SILVER Cards in Germany                     GOLD Germany, SILVER Germany
        Low Satisfaction Score      Any segment with avg satisfaction score < 3
        Low Satisfaction Score      Any segment with avg satisfaction score < 3
